# models

> Data shapes for the transcription pipeline: run configuration + the run-manifest result containers.

The run manifest is the pipeline's durable output record: which sources were processed, how they were segmented, and where each segment's transcription landed (capability data DBs remain the authoritative text store; the manifest records the run's shape + provenance pointers). It is a deliberate proto-bundle — the CR-20 provenance-bundle infrastructure is expected to absorb/replace it.

In [ ]:
#| default_exp models

In [ ]:
#| export
import json
import time
import uuid
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

In [ ]:
#| export
@dataclass
class PipelineConfig:
    """Configuration for one transcription pipeline run."""
    vad_plugin: str = "cjm-media-plugin-silero-vad"               # VAD capability instance id
    ffmpeg_plugin: str = "cjm-media-plugin-ffmpeg"                # Convert/segment capability instance id
    transcriber_plugins: List[str] = field(                       # Transcription capability instance ids (one or more; stage-5 dual-transcriber)
        default_factory=lambda: ["cjm-transcription-plugin-whisper"])
    graph_plugin: Optional[str] = None   # Graph-storage capability for Source/AudioSegment/Transcript emission (None = no emission)
    graph_db_path: Optional[str] = None  # Explicit graph DB path override (caller-wins config, C8/F10)
    # Opt-in audio preprocessing (stage 8 — Demucs source separation is the first
    # family). When set, each ~5-min segment is preprocessed BEFORE the model-input
    # convert: full-band segment -> separate_vocals -> convert -> transcribe / (decomp) VAD+FA.
    # The slot is FAMILY-AGNOSTIC: it routes through the task channel by
    # (preprocessing_task, preprocessing_method), so a future preprocessing family
    # (e.g. speech-enhancement) drops in by changing those, not the pipeline.
    preprocessing_plugin: Optional[str] = None        # Preprocessing capability instance id (None = preprocessing OFF)
    preprocessing_task: str = "source_separation"     # Task-channel task for the preprocessing step
    preprocessing_method: str = "separate_vocals"     # Task-channel method for the preprocessing step
    max_segment_duration: float = 300.0  # Wall-clock cap per segment in seconds (pre-emptive cuts)
    sample_rate: int = 16000             # Model-input sample rate for the per-segment convert step
    channels: int = 1                    # Model-input channel count
    force: bool = False                  # Bypass capability-side caches (VAD + transcription + preprocessing)
    assume_yes: bool = False             # Auto-accept HITL seams (headless / corpus-generation mode)

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict snapshot for the run manifest
        """Serialize to a plain dict."""
        return asdict(self)

In [ ]:
#| export
@dataclass
class SegmentRecord:
    """One segment of a source audio file, with per-transcriber transcripts.

    Manifest schema 0.2.0: the single text/job_id pair became `transcripts`
    keyed by transcriber capability name — transcription emits SYMMETRIC
    variants; the authority designation is the decomp consumer's choice
    (stage-5 ratified design). `model_input_hash` content-addresses the
    model-input WAV (the audio of record, E14) for graph emission identity."""
    index: int              # 0-based position within the source
    start: float            # Segment start in source-audio seconds
    end: float              # Segment end in source-audio seconds
    duration: float         # Wall-clock segment duration in seconds
    segment_path: str       # Cut audio file (source codec) from ffmpeg `segment_audio`
    model_input_path: str   # Model-ready WAV from the per-segment `convert` step
    model_input_hash: str = ""  # Content hash over the model-input WAV ("algo:hexdigest")
    transcripts: Dict[str, Dict[str, Any]] = field(default_factory=dict)  # transcriber -> {job_id, text, metadata}

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form for the run manifest
        """Serialize to a plain dict."""
        return asdict(self)

In [ ]:
#| export
@dataclass
class SourceResult:
    """Pipeline result for one source audio file."""
    source_path: str        # Original input audio path
    duration: float         # Source duration in seconds
    vad_chunk_count: int    # Number of speech chunks VAD detected
    batch_key: str          # ffmpeg `segment_audio` batch key linking the cut files
    content_hash: str = ""  # Content hash over the source file (Source node identity input)
    segments: List[SegmentRecord] = field(default_factory=list)  # Ordered transcribed segments
    chain: List[str] = field(default_factory=list)  # Preprocessing chain that produced the model-inputs ([] = raw convert-only); AudioRendition identity input — extenders recompute the rendition id from it
    graph: Optional[Dict[str, Any]] = None  # Emission record: {source_node_id, nodes_added, nodes_verified, edges_added} (None = not emitted)

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form for the run manifest
        """Serialize to a plain dict with nested segments."""
        return {
            "source_path": self.source_path,
            "duration": self.duration,
            "vad_chunk_count": self.vad_chunk_count,
            "batch_key": self.batch_key,
            "content_hash": self.content_hash,
            "segments": [s.to_dict() for s in self.segments],
            "chain": list(self.chain),
            "graph": self.graph,
        }

In [ ]:
#| export
@dataclass
class RunManifest:
    """Durable record of one pipeline run (proto-bundle; see CR-20).

    Schema 0.3.0 (AudioRendition era): per-source `chain` records the
    preprocessing chain that produced the model-inputs ([] = raw convert-only),
    so a downstream extender can RECOMPUTE the deterministic AudioRendition node
    id (and the Transcript/Segment ids keyed on it) with no search. Builds on
    0.2.0's per-segment `transcripts` keyed by transcriber + source
    `content_hash` + per-segment `model_input_hash` + plugin `config_hash`."""
    run_id: str                       # Unique run identifier
    created_at: float                 # Unix timestamp at run start
    config: Dict[str, Any]            # PipelineConfig snapshot
    plugins: Dict[str, Dict[str, Any]] = field(default_factory=dict)  # instance_id -> {name, version, db_path, config_hash}
    sources: List[SourceResult] = field(default_factory=list)         # Per-source results, input order
    graph: Optional[Dict[str, Any]] = None  # Emission target: {plugin, db_path} (None = no emission this run)

    FORMAT: str = field(default="cjm-transcription-core/run-manifest", repr=False)  # Manifest format tag
    VERSION: str = field(default="0.3.0", repr=False)                               # Manifest schema version

    def to_dict(self) -> Dict[str, Any]:  # Plain-dict form for JSON serialization
        """Serialize to a plain dict with nested sources."""
        return {
            "format": self.FORMAT,
            "version": self.VERSION,
            "run_id": self.run_id,
            "created_at": self.created_at,
            "config": self.config,
            "plugins": self.plugins,
            "sources": [s.to_dict() for s in self.sources],
            "graph": self.graph,
        }

    def save(
        self,
        path: Union[str, Path],  # Destination JSON file (parent dirs created)
    ) -> Path:  # The written path
        """Write the manifest as pretty-printed JSON."""
        out = Path(path)
        out.parent.mkdir(parents=True, exist_ok=True)
        out.write_text(json.dumps(self.to_dict(), indent=2))
        return out

In [ ]:
#| export
def new_run_id() -> str:  # e.g. "run_20260607_153000_1a2b3c4d"
    """Generate a unique, sortable run id."""
    return f"run_{time.strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

In [ ]:
# Quick shape check (no plugins involved)
cfg = PipelineConfig()
m = RunManifest(run_id=new_run_id(), created_at=time.time(), config=cfg.to_dict())
assert m.to_dict()["format"] == "cjm-transcription-core/run-manifest"
assert m.to_dict()["version"] == "0.3.0"
assert m.to_dict()["config"]["max_segment_duration"] == 300.0
assert m.to_dict()["config"]["transcriber_plugins"] == ["cjm-transcription-plugin-whisper"]
assert m.to_dict()["graph"] is None
# preprocessing is OFF by default; the family-agnostic slot defaults to source_separation
assert m.to_dict()["config"]["preprocessing_plugin"] is None
assert m.to_dict()["config"]["preprocessing_task"] == "source_separation"
assert m.to_dict()["config"]["preprocessing_method"] == "separate_vocals"

# 0.2.0 per-transcriber segment shape
rec = SegmentRecord(index=0, start=0.0, end=10.0, duration=10.0,
                    segment_path="/cuts/s0.mp3", model_input_path="/cache/s0.wav",
                    model_input_hash="sha256:wav",
                    transcripts={"whisper": {"job_id": "j1", "text": "hi", "metadata": {}}})
d = rec.to_dict()
assert d["transcripts"]["whisper"]["text"] == "hi" and d["model_input_hash"] == "sha256:wav"
src = SourceResult(source_path="/a.mp3", duration=10.0, vad_chunk_count=3,
                   batch_key="bk", content_hash="sha256:src", segments=[rec])
assert src.to_dict()["content_hash"] == "sha256:src" and src.to_dict()["graph"] is None
# chain defaults to [] (raw convert-only); a preprocessed run records its chain
assert src.to_dict()["chain"] == []
src_vox = SourceResult(source_path="/a.mp3", duration=10.0, vad_chunk_count=3, batch_key="bk",
                       content_hash="sha256:src", segments=[rec],
                       chain=["source_separation:cjm-media-plugin-demucs@cfg"])
assert src_vox.to_dict()["chain"] == ["source_separation:cjm-media-plugin-demucs@cfg"]
m.to_dict()["run_id"]